In [1]:
import biom
import pandas as pd
import numpy as np

In [2]:
table = biom.load_table('emp_deblur_90bp.release1.biom')
#meta = pd.read_csv('emp_qiime_mapping_release1.tsv', sep='\t', index_col=0, low_memory=False) metadata dei sample

print('Campioni:', table.shape[1])
print('OTU/Phyla:', table.shape[0])
print('Primi 5 campioni:', table.ids(axis='sample')[:5])
print('Primi 5 phyla:', table.ids(axis='observation')[:5])

Campioni: 26181
OTU/Phyla: 317314
Primi 5 campioni: ['1736.B10.0610' '1721.S8B'
 '1883.2011.086.Crump.Artic.LTREB.main.lane3.NoIndex'
 '1883.2008.269.Crump.Artic.LTREB.main.lane2.NoIndex'
 '755.SSFA.L1.D1.31.05.11.lane1.NoIndex.L001']
Primi 5 phyla: ['TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCACGTAGGCGGCTGTTTAAGCTAGCTGTGAAAGCCCCGGGCTTAAC'
 'TACAGAGGTCCCAAGCGTTGTTCGGATTCATTGGGCGTAAAGGGCTCGTAGGTGGCCAACTAAGTCAGACGTGAAATCCCTCGGCTTAAC'
 'TACGAAGGGTGCAAGCGTTGTTCGGAATAACTGGGCGTAAAGCGCACGTAGGCGGGTCCGTGTGTCGGTTGTGAAATCCCTGGGCTCAAC'
 'TACGGAGTGTGCAAGCGTTACTCGGAATCACTGGGCATAAAGAGCACGTAGGCGGGTCACCAAGTCAGCCGTGAAAGCCCCCGGCCCAAC'
 'TACGAGAGGTCCAAACGTTATTCGGAATTACTGGGCTTAAAGAGTTCGTAGGCGGCTAAGTAAGTGGGATGTGAAAGCCCTCGGCTCAAC']


In [3]:
def get_phylum(otu_id, meta):
    tax = meta['taxonomy']
    if len(tax) >= 2 and tax[1] != '':
        return tax[1]
    return 'p__Unknown'

In [4]:
# Collassa la tabella biom per phylum
table_phylum = table.collapse(
    get_phylum, 
    axis='observation',
    norm=False
)

In [5]:
# Converti in dataframe
df_phylum = table_phylum.to_dataframe(dense=True)  # righe = phyla, colonne = campioni
print(df_phylum.head(5))

                   1736.B10.0610  1721.S8B  \
p__Proteobacteria          343.0   15888.0   
p__Unknown                3287.0     996.0   
p__Planctomycetes          117.0    2258.0   
p__Firmicutes            28047.0    7381.0   
p__Bacteroidetes         16836.0    1352.0   

                   1883.2011.086.Crump.Artic.LTREB.main.lane3.NoIndex  \
p__Proteobacteria                                            11479.0    
p__Unknown                                                     242.0    
p__Planctomycetes                                             1401.0    
p__Firmicutes                                                    4.0    
p__Bacteroidetes                                              3298.0    

                   1883.2008.269.Crump.Artic.LTREB.main.lane2.NoIndex  \
p__Proteobacteria                                            16815.0    
p__Unknown                                                     226.0    
p__Planctomycetes                                            1516

In [6]:
# Trasponi e normalizza
X_sb = df_phylum.T
X_sb = X_sb.div(X_sb.sum(axis=1), axis=0)
print(X_sb.head())

                                                    p__Proteobacteria  \
1736.B10.0610                                                0.006497   
1721.S8B                                                     0.279689   
1883.2011.086.Crump.Artic.LTREB.main.lane3.NoIndex           0.414344   
1883.2008.269.Crump.Artic.LTREB.main.lane2.NoIndex           0.217673   
755.SSFA.L1.D1.31.05.11.lane1.NoIndex.L001                   0.516513   

                                                    p__Unknown  \
1736.B10.0610                                         0.062262   
1721.S8B                                              0.017533   
1883.2011.086.Crump.Artic.LTREB.main.lane3.NoIndex    0.008735   
1883.2008.269.Crump.Artic.LTREB.main.lane2.NoIndex    0.002926   
755.SSFA.L1.D1.31.05.11.lane1.NoIndex.L001            0.010898   

                                                    p__Planctomycetes  \
1736.B10.0610                                                0.002216   
1721.S8B          

In [7]:
def shannon_entropy(row):
    p = row[row > 0]  # ignora zeri
    return -np.sum(p * np.log(p))

In [8]:
# Diversità dei campioni
H_samples = X_sb.apply(shannon_entropy, axis=1)
print("Entropia campioni:")
print(H_samples.head())

Entropia campioni:
1736.B10.0610                                         1.259346
1721.S8B                                              2.238489
1883.2011.086.Crump.Artic.LTREB.main.lane3.NoIndex    1.466594
1883.2008.269.Crump.Artic.LTREB.main.lane2.NoIndex    1.861829
755.SSFA.L1.D1.31.05.11.lane1.NoIndex.L001            1.691105
dtype: float64


In [9]:
# Ubiquità dei phyla
H_phyla = X_sb.apply(shannon_entropy, axis=0)
print("Entropia phyla:")
print(H_phyla.head())

Entropia phyla:
p__Proteobacteria    7277.872933
p__Unknown           1406.460114
p__Planctomycetes    1492.025591
p__Firmicutes        3661.961249
p__Bacteroidetes     5539.499641
dtype: float64
